# Notebook 8: Function Calling & Tool Use

LLMs can use **tools** — external functions, APIs, and databases. In this notebook you'll learn:
- How LLMs call functions by generating structured output
- The tool-use loop: generate → execute → feed back
- How RLM's `llm_query()` is a special case of tool use (the tool is *the model itself*)
- Why RLMs are more flexible than predefined tool sets

## The Connection to RLMs

`llm_query()` follows the exact same pattern as any tool call:
1. Model decides to use a tool → calls `llm_query("sub-question")`
2. System executes the tool → another LLM call runs
3. Result returns to the model → answer flows back to sandbox code

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")
import json, re

from rlm_core import LLMClient, RLMEngine, Sandbox
from rlm_core.visualizer import tree_to_text

class SimulatedClient:
    def __init__(self):
        self.call_count = 0
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
    
    def completion(self, prompt, **kwargs):
        self.call_count += 1
        self.total_prompt_tokens += len(prompt.split())
        self.total_completion_tokens += 15
        
        class Result:
            def __init__(self, text, pt, ct):
                self.text, self.prompt_tokens, self.completion_tokens = text, pt, ct
        
        # Simple routing based on prompt content
        if "calculator" in prompt.lower() or "compute" in prompt.lower():
            text = '{"tool": "calculator", "input": "15 * 24 + 7"}'
        elif "search" in prompt.lower() or "find" in prompt.lower():
            text = '{"tool": "search", "input": "population of France"}'
        elif "secret" in prompt.lower():
            text = '''import re
match = re.search(r'secret code is ([A-Z0-9-]+)', context)
FINAL(match.group(1) if match else "Not found")'''
        else:
            text = '{"tool": "search", "input": "general information"}'
        
        return Result(text, len(prompt.split()), 15)

client = SimulatedClient()
print("Ready!")

## Step 1: Basic Function Calling

The simplest pattern: tell the LLM about available tools, have it output a structured call, parse and execute it.

In [ ]:
# Define tools the LLM can use
tools = {
    "calculator": lambda expr: str(eval(expr)),  # Simple calculator
    "search": lambda query: f"[Search result for '{query}': France has a population of ~68 million]",
}

def tool_calling_llm(client, question, available_tools):
    """LLM that uses tools via structured JSON output."""
    tool_descriptions = "\n".join(f"- {name}: {func.__doc__ or 'No description'}" 
                                   for name, func in available_tools.items())
    
    prompt = f"""You have these tools available:
{tool_descriptions}

To use a tool, respond with JSON: {{"tool": "name", "input": "argument"}}
To give a final answer, respond with JSON: {{"answer": "your answer"}}

Question: {question}"""
    
    response = client.completion(prompt)
    
    try:
        call = json.loads(response.text)
        if "tool" in call:
            tool_name = call["tool"]
            tool_input = call["input"]
            print(f"  Tool call: {tool_name}({tool_input!r})")
            
            if tool_name in available_tools:
                result = available_tools[tool_name](tool_input)
                print(f"  Tool result: {result}")
                return result
            else:
                return f"Unknown tool: {tool_name}"
        elif "answer" in call:
            return call["answer"]
    except json.JSONDecodeError:
        return response.text

# Demo
print("=== Tool-Calling LLM ===")
answer = tool_calling_llm(client, "What is 15 * 24 + 7?", tools)
print(f"Final answer: {answer}")

## Step 2: The Tool-Use Loop

Real tool use involves multiple rounds: the model calls a tool, gets the result, then decides whether to call another tool or give a final answer.

In [ ]:
def tool_use_loop(client, question, available_tools, max_rounds=5):
    """Multi-round tool use: call tools until the model gives a final answer."""
    history = [f"Question: {question}"]
    
    for round_num in range(max_rounds):
        prompt = "\n".join(history) + "\n\nUse a tool or give a final answer (JSON):"
        response = client.completion(prompt)
        
        try:
            call = json.loads(response.text)
        except json.JSONDecodeError:
            history.append(f"Response: {response.text}")
            continue
        
        if "answer" in call:
            print(f"  Round {round_num + 1}: Final answer: {call['answer']}")
            return call["answer"], round_num + 1
        
        if "tool" in call:
            tool_name = call["tool"]
            tool_input = call["input"]
            print(f"  Round {round_num + 1}: {tool_name}({tool_input!r})")
            
            if tool_name in available_tools:
                result = available_tools[tool_name](tool_input)
                history.append(f"Tool call: {tool_name}({tool_input!r}) → {result}")
            else:
                history.append(f"Error: Unknown tool {tool_name}")
    
    return "Max rounds reached", max_rounds

print("=== Tool-Use Loop ===")
answer, rounds = tool_use_loop(client, "What is the population of France times 2?", tools)
print(f"\nResolved in {rounds} round(s)")

## Step 3: llm_query() as Self-Tool-Use

Now here's the key insight: RLM's `llm_query()` follows the **exact same pattern**, but the tool is another LLM call:

| Traditional Tool Use | RLM Self-Tool-Use |
|---------------------|-------------------|
| Model calls `calculator("2+2")` | Model calls `llm_query("What is 2+2?")` |
| Calculator returns `"4"` | Sub-LLM returns `"4"` |
| Model uses result | Model uses result |

The difference: `llm_query()` is **infinitely flexible**. Instead of being limited to predefined tools, the model can ask ANY question.

In [ ]:
# Side by side: tool calling vs RLM

print("=== Traditional Tool Calling ===")
print("Model thinks: 'I need to calculate something'")
print("Model outputs: {\"tool\": \"calculator\", \"input\": \"15 * 24\"}")
result = tools["calculator"]("15 * 24")
print(f"Tool returns: {result}")
print(f"Model uses result to answer.")

print("\n=== RLM Self-Tool-Use ===")
print("Model thinks: 'I need to find info in a large document'")
print("Model writes code:")
print('  chunks = [context[:1000], context[1000:2000]]')
print('  for chunk in chunks:')
print('      answer = llm_query(f"Find the secret in: {chunk[:50]}...")')
print("Each llm_query() call → full LLM execution → result flows back")
print("\nThe model is using ITSELF as a universal tool!")

## Step 4: Why RLMs Are More Flexible

Traditional tool use requires **predefined tools**. The developer must anticipate what the model needs.

RLMs let the model **write its own tools** at runtime:

In [ ]:
# Traditional: limited to predefined tools
print("Traditional tool-calling LLM:")
print("  Can only use: calculator, search")
print("  What if it needs regex? Text splitting? JSON parsing?")
print("  → Developer must add more tools")

print("\nRLM approach:")
print("  Model writes whatever code it needs:")

sb = Sandbox(variables={"context": "Price: $42.99, Quantity: 3"})
result = sb.execute("""
import re
# Extract price and quantity (no predefined tool needed!)
price = float(re.search(r'\\$(\\d+\\.\\d+)', context).group(1))
qty = int(re.search(r'Quantity: (\\d+)', context).group(1))
total = price * qty
print(f"Total: ${total:.2f}")
""")
print(f"  Sandbox output: {result.stdout.strip()}")
print("\n  The model created its OWN tool (regex extraction + math) on the fly!")

## Step 5: Comparison — Same Task, Both Approaches

In [ ]:
with open("../data/samples/needle_haystack.txt") as f:
    document = f.read()

# Approach 1: Tool-calling with a search tool
search_tool = {
    "search": lambda query: document[:500] + "..."  # Simplified: returns first 500 chars
}
print("=== Tool-Calling Approach ===")
answer1 = tool_calling_llm(client, "Find the secret code in the document", search_tool)
print(f"Answer: {answer1[:100]}...")
print("Limitation: search tool only returns first 500 chars — might miss the needle!")

# Approach 2: RLM
print("\n=== RLM Approach ===")
engine = RLMEngine(client=client, max_depth=3)
result = engine.run("What is the secret code?", document)
print(f"Answer: {result.answer}")
print(f"Strategy: Model wrote code to search the ENTIRE document with regex")
print(tree_to_text(result.root_node))

## Exercise

Build a tool-calling LLM with 3 tools (calculator, search, weather), then show the same tasks solved by RLM without any predefined tools.

In [ ]:
# Exercise: Add a weather tool and try both approaches

my_tools = {
    "calculator": lambda expr: str(eval(expr)),
    "search": lambda q: f"[Result for '{q}': Information found]",
    "weather": lambda city: f"[Weather in {city}: 22C, sunny]",
}

# Tool-calling approach
print("=== Your Tool-Calling LLM ===")
answer = tool_calling_llm(client, "What's the weather in Paris?", my_tools)
print(f"Answer: {answer}")

# RLM approach (no predefined tools needed)
print("\n=== Your RLM ===")
result = engine.run(
    "What is the weather in Paris?",
    "Weather data: Paris - 22C sunny, London - 15C cloudy, Tokyo - 28C humid"
)
print(f"Answer: {result.answer}")

## Key Takeaways

1. **Function calling** lets LLMs use external tools via structured output (JSON)
2. **The tool-use loop** enables multi-step reasoning with tool calls
3. **`llm_query()` is self-tool-use** — the model uses itself as a universal tool
4. **RLMs are more flexible** — the model writes its own tools at runtime via code
5. **Traditional tools are predefined and limited**; RLM tools are created on-the-fly
6. **The tradeoff:** Tool calling is simpler and more predictable; RLMs are more powerful but less controlled

## What's Next?

The **Comparisons Track** begins! Notebook 9 compares RLM vs RAG (Retrieval-Augmented Generation) — the dominant approach for LLMs + external data.